# Dónde construir: estaciones para una alcaldía nueva

## Contexto

Una nueva administración acaba de asumir la Alcaldía de Medellín. Dentro del plan de desarrollo se aprobó una partida para construir estaciones que atiendan tres tipos de incidentes:

| Tipo de incidente | Estación que lo atiende |
|---|---|
| Hurto | CAI de policía |
| Accidente de tránsito | Estación de tránsito y grúas |
| Incendio o emergencia | Estación de bomberos |

La Secretaría de Planeación entrega un histórico de incidentes de los últimos años, la red vial de la ciudad y una lista de lotes disponibles donde se podría construir. Cada estación tiene un costo que depende de su tipo y de la zona del lote, y solo atiende con oportunidad los incidentes de su tipo que estén a cierta distancia por las calles.

El presupuesto no alcanza para cubrir toda la ciudad. Tu equipo debe decidir **qué estaciones construir y en qué lotes**, de manera que la ciudad quede lo mejor atendida posible sin exceder el presupuesto, y presentar la propuesta al concejo con un mapa, el costo y los casos que se cubrirían.

La red vial corresponde a las calles reales de Medellín. El histórico de incidentes, los lotes y los costos son simulados para el curso.

## Los datos

Cada instancia está en una carpeta con cinco archivos:

- **`nodos.csv`**: intersecciones y puntos de la red. Columnas `id`, `lat`, `lon`.
- **`aristas.csv`**: tramos de calle. Columnas `origen`, `destino`, `longitud_m`, `nombre`. Cada fila es un tramo que se puede recorrer **solo de `origen` a `destino`**. Una calle de doble sentido aparece dos veces, una por cada dirección. Algunos tramos no tienen nombre.
- **`lotes.csv`**: lotes disponibles. Columnas `lote` (código), `nodo` (intersección donde está el lote) y `factor_zona`.
- **`incidentes.csv`**: histórico. Columnas `id`, `nodo` (intersección donde ocurrió), `tipo` (`hurto`, `accidente` o `incendio`) y `fecha_hora`.
- **`parametros.json`**: presupuesto total y, para cada tipo de incidente, el nombre de la estación, su costo base, su radio de cobertura y su peso de prioridad.

Los parámetros de las tres instancias son los mismos salvo el presupuesto:

| Tipo | Estación | Costo base (millones) | Radio de cobertura | Peso |
|---|---|---|---|---|
| hurto | CAI | 1.000 | 1.000 m | 1 |
| accidente | Tránsito y grúas | 2.000 | 2.000 m | 3 |
| incendio | Bomberos | 5.000 | 3.000 m | 5 |

### Las tres instancias

| Instancia | Descripción | Nodos | Tramos | Incidentes | Lotes | Presupuesto (millones) |
|---|---|---|---|---|---|---|
| `pequena` | Barrio ficticio | 9 | 22 | 18 | 5 | 6.000 |
| `comuna` | Recorte de la red real alrededor de Laureles | 2.893 | 6.782 | 913 | 15 | 12.000 |
| `ciudad` | Red vial completa de Medellín | 27.568 | 67.330 | 5.300 | 80 | 40.000 |

### Descripción del barrio de la instancia `pequena`

El barrio es una cuadrícula de 3 × 3 intersecciones numeradas de 0 a 8. Vista como un mapa con el norte hacia arriba:

```
 6 ── 7 ── 8        fila de arriba:  Calle 3
 │    │    ↑
 3 ── 4 →─ 5        fila del medio:  Calle 2
 │    │    │
 0 ── 1 ── 2        fila de abajo:   Calle 1

Carrera A (0-3-6)   Carrera B (1-4-7)   Carrera C (2-5-8)
```

Todas las calles son de doble sentido excepto dos tramos: en la Calle 2 solo se puede ir **de 4 a 5**, y en la Carrera C solo se puede ir **de 5 a 8**. Cada tramo mide entre 900 m y 990 m; las longitudes exactas están en `aristas.csv`.

Los lotes disponibles están en las intersecciones 1, 3, 5, 6 y 8:

| lote | nodo | factor_zona |
|---|---|---|
| L1 | 1 | 1.1 |
| L2 | 3 | 1.5 |
| L3 | 5 | 1.4 |
| L4 | 6 | 1.5 |
| L5 | 8 | 1.5 |

Los 18 incidentes del histórico se distribuyen así (H = hurtos, A = accidentes, I = incendios):

| Nodo | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|---|
| Incidentes | — | H1 | H1 A2 I1 | H3 | H2 A1 | A1 | A1 I1 | H1 | H1 A1 I1 |

## Reglas del modelo

1. **Costo.** Construir una estación de un tipo en un lote cuesta `costo_base del tipo × factor_zona del lote`, redondeado al millón más cercano. Por ejemplo, un CAI en el lote L1 de la instancia `pequena` cuesta 1.000 × 1.1 = 1.100 millones.
2. **Un lote, una estación.** En cada lote se construye a lo sumo una estación. Sí se pueden construir varias estaciones del mismo tipo en lotes distintos.
3. **Presupuesto.** La suma de los costos de las estaciones construidas no puede superar el presupuesto de la instancia.
4. **Cobertura.** Un incidente queda cubierto si existe una estación **de su mismo tipo** tal que la distancia más corta por la red vial **desde el nodo de la estación hasta el nodo del incidente** es menor o igual al radio de cobertura de ese tipo. La distancia se mide respetando el sentido de los tramos, porque es el vehículo de la estación el que se desplaza hasta el incidente.
5. **Valor de una propuesta.** Es la suma de los pesos de los incidentes cubiertos. Un incidente cubierto por varias estaciones cuenta una sola vez. Por ejemplo, una propuesta que cubre 10 hurtos, 4 accidentes y 1 incendio vale 10 × 1 + 4 × 3 + 1 × 5 = 27.

### Supuestos

- La red vial es una aproximación: se retiraron ciclorrutas, senderos y pasos peatonales identificables por su nombre.
- Los vehículos de emergencia pueden circular por los carriles de Metroplús, que hacen parte de la red.
- Todos los incidentes de un mismo tipo tienen la misma prioridad, sin importar su fecha.

## Restricciones de implementación

- La solución se escribe en Python.
- Se pueden usar `csv`, `json` o `pandas` **solo para leer los archivos**, y `heapq`.
- Para el mapa y la interfaz gráfica se puede usar cualquier librería de visualización o de interfaces (por ejemplo `folium`, `matplotlib`, `streamlit` o `tkinter`), siempre que no se use para calcular rutas, distancias ni combinaciones.
- No se permite usar `networkx`, `osmnx`, `scipy`, `itertools` ni ninguna librería que calcule rutas, distancias en grafos o combinaciones.

## Cronograma y entregables

| Semana | Entrega |
|---|---|
| 1 | Parte 1 |
| 2 | Parte 2 |
| 3 | Parte 3 |
| 4 | Parte 4 y sustentación |

Cada entrega se hace sobre este mismo notebook. Las partes posteriores pueden reutilizar el código de las anteriores.

### Sustentación

En la sustentación se entregará a cada equipo una nueva distribución de lotes y un nuevo histórico de incidentes, con el mismo formato de archivos descrito en la sección "Los datos". El equipo deberá ejecutar su solución sobre esos datos durante la sustentación y ajustarla si es necesario, así que conviene que el código lea cualquier instancia desde sus archivos y no dependa de valores particulares de las instancias entregadas.

### Nota: feria de proyectos de ingeniería

Los equipos que quieran presentar esta práctica en la feria de proyectos de ingeniería deben, además de todo lo anterior, desarrollar una **aplicación interactiva con interfaz gráfica de usuario**, por ejemplo una que permita elegir la instancia, cambiar el presupuesto o los pesos de prioridad y ver la propuesta resultante sobre el mapa. Sin esta aplicación el proyecto no podrá competir.

El equipo ganador de la feria quedará **eximido del examen final**.

## Parte 1. Alcance de cada estación posible

Para una instancia cualquiera, calcula para **cada lote y cada tipo de estación**:

- el costo de construir esa estación en ese lote, y
- la lista de `id` de los incidentes que esa estación cubriría por sí sola, según la regla 4.

### Entrada

El nombre de una instancia (`pequena`, `comuna` o `ciudad`). El programa lee los cinco archivos de su carpeta.

### Salida

Para cada par (lote, tipo): su costo en millones y la lista ordenada de `id` de los incidentes que cubre.

### Restricción adicional

El cálculo para la instancia `ciudad`, con sus 80 lotes y 3 tipos, debe terminar en menos de un minuto.

In [ ]:
casos_ej_001_parte1 = [
    {
        "entrada": {"instancia": "pequena", "lote": "L3", "tipo": "accidente"},
        "salida": {"costo": 2800, "incidentes_cubiertos": [1, 2, 4, 6]},
    },
    {
        "entrada": {"instancia": "pequena", "lote": "L2", "tipo": "hurto"},
        "salida": {"costo": 1500, "incidentes_cubiertos": [9, 10, 11, 12, 13]},
    },
]

## Parte 2. Propuestas construidas paso a paso

### 2.1 Heurística base

La Secretaría de Planeación usa hoy el siguiente procedimiento para armar una propuesta:

1. Empezar sin estaciones y con todo el presupuesto disponible.
2. Considerar todas las estaciones posibles (lote, tipo) que cumplen dos condiciones: el lote todavía está libre y su costo no supera el presupuesto restante.
3. De ellas, elegir la que **agrega más valor**, es decir, la que tiene mayor suma de pesos de los incidentes que cubre y que **todavía no están cubiertos** por las estaciones ya elegidas. Si hay empate, se elige la de menor código de lote; si el empate es dentro del mismo lote, se prefiere en el orden hurto, accidente, incendio.
4. Si ninguna estación posible agrega valor, terminar. Si no, construirla, descontar su costo, marcar el lote como ocupado y volver al paso 2.

Implementa la heurística base y aplícala a las tres instancias.

### 2.2 Heurísticas propias

Diseña al menos una heurística propia que también construya la propuesta eligiendo una estación a la vez, y que obtenga un valor mayor que la base en al menos una de las tres instancias. Aplícala a las tres instancias.

Para cada heurística, la base incluida, construye a mano una instancia pequeña (en el mismo formato de archivos, con no más de 6 nodos) en la que la heurística **no** obtenga la mejor propuesta posible. Explica qué decisión de la heurística la lleva por mal camino en ese caso.

### Salida

Para cada heurística e instancia: las estaciones elegidas (lote y tipo, ordenadas por código de lote), el costo total, la cantidad de casos cubiertos de cada tipo y el valor de la propuesta.

In [ ]:
casos_ej_001_parte2 = [
    {
        "entrada": {"instancia": "pequena", "heuristica": "base"},
        "salida": {
            "estaciones": [("L1", "incendio")],
            "costo_total": 5500,
            "casos_cubiertos": {"hurto": 0, "accidente": 0, "incendio": 3},
            "valor": 15,
        },
    },
    {
        "entrada": {"instancia": "comuna", "heuristica": "base"},
        "salida": {
            "estaciones": [
                ("L05", "accidente"), ("L06", "hurto"), ("L08", "accidente"),
                ("L09", "incendio"), ("L13", "hurto"), ("L14", "hurto"),
                ("L15", "accidente"),
            ],
            "costo_total": 11950,
            "casos_cubiertos": {"hurto": 486, "accidente": 296, "incendio": 31},
            "valor": 1529,
        },
    },
]

## Parte 3. La mejor propuesta posible

Obtén la propuesta de **mayor valor posible** que respete todas las reglas, para las instancias `pequena` y `comuna`. Tu programa debe garantizar que no existe una propuesta mejor, no solo encontrar una buena.

Luego compara en una tabla, para las tres instancias, el valor y el tiempo de ejecución de la heurística base, de tus heurísticas propias y de la mejor propuesta posible. En la instancia `ciudad`, explica con un cálculo por qué no reportas la mejor propuesta posible.

### Salida

Para `pequena` y `comuna`: las estaciones de la mejor propuesta (ordenadas por código de lote), su costo total, los casos cubiertos de cada tipo, su valor y el tiempo que tardó en obtenerse. Además, la tabla comparativa.

In [ ]:
casos_ej_001_parte3 = [
    {
        "entrada": {"instancia": "pequena"},
        "salida": {
            "estaciones": [("L1", "hurto"), ("L3", "accidente"), ("L4", "hurto")],
            "costo_total": 5400,
            "casos_cubiertos": {"hurto": 8, "accidente": 4, "incendio": 0},
            "valor": 20,
        },
    },
]

## Parte 4. Propuesta para el concejo

Elige la propuesta que recomendarías para la instancia `ciudad`, obtenida con cualquiera de tus heurísticas, y preséntala con:

1. **Un mapa** de la ciudad, hecho con `folium` o con cualquier otra librería, donde se distingan los lotes no usados, las estaciones construidas según su tipo, y los incidentes cubiertos y no cubiertos. Mostrar el resultado de forma gráfica sobre un mapa es obligatorio para todos los equipos, participen o no en la feria.
2. **Una tabla** con cada estación construida: lote, tipo, costo y casos que cubre. Al final, el costo total frente al presupuesto, los casos cubiertos de cada tipo y el valor de la propuesta.
3. **Un párrafo** que justifique por qué recomiendas esa propuesta y no otra de las que obtuviste.

Repite el mapa y la tabla para la mejor propuesta posible de la instancia `comuna`.